# 4. CellRank Trajectory Analysis

This notebook performs cell fate trajectory analysis using CellRank and moscot (optimal transport).

**Input:** Annotated EC h5ad file with subcluster annotations

**Output:** Fate probability matrices for WT and Het conditions

## Overview

CellRank uses optimal transport (via moscot) to:
1. Couple cells across timepoints (Week 0 → Week 12)
2. Compute transition probabilities between EC subtypes
3. Predict cell fate probabilities

## EC Subtypes
| Index | Label | Description |
|-------|-------|-------------|
| 0 | EC1.1 | Arterial EC |
| 1 | EC1.2 | Inflammatory EC |
| 2 | EC1.3 | Mesenchymal-like EC |
| 3 | EC2 | Capillary EC |
| 4 | EC3 | Lymphatic EC |

## Setup

In [ ]:
from moscot.problems.time import TemporalProblem

import cellrank as cr
import scanpy as sc
from cellrank.kernels import RealTimeKernel
import numpy as np
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt

import warnings
sc.settings.set_figure_params(frameon=False, dpi=100)
cr.settings.verbosity = 2
warnings.simplefilter("ignore", category=UserWarning)

print(f"CellRank version: {cr.__version__}")

## Configuration

In [ ]:
# Paths - UPDATE THESE
data_dir = "./"
output_dir = "./"

# Input file: Annotated EC h5ad (exported from Seurat)
input_file = "EC_annotated.h5ad"

# Output file for fate probability results
output_file = "cellrank_fate_probabilities.xlsx"

# EC subtype mapping
EC_LABELS = {
    0: "EC1.1",
    1: "EC1.2", 
    2: "EC1.3",
    3: "EC2",
    4: "EC3"
}

# Group mapping
# 0 = WT_Week0, 1 = Het_Week0, 2 = WT_Week12, 3 = Het_Week12
GROUP_MAPPING = {
    0: "WT_Week0",
    1: "Het_Week0",
    2: "WT_Week12",
    3: "Het_Week12"
}

## Load Data

In [ ]:
import os

adata_orig = sc.read_h5ad(os.path.join(data_dir, input_file))

print(f"Loaded: {adata_orig.n_obs} cells, {adata_orig.n_vars} genes")
print(f"\nObservations columns: {list(adata_orig.obs.columns)}")
print(f"\nGroup distribution:")
print(adata_orig.obs["group"].value_counts())

## CellRank Analysis Function

This function:
1. Subsets data by genotype (WT or Het)
2. Runs moscot optimal transport to couple cells across timepoints
3. Computes transition matrix using RealTimeKernel
4. Calculates fate probabilities for each EC subtype
5. Returns mean fate probabilities per initial subtype

In [ ]:
def run_cellrank_analysis(adata_orig, genotype, conn_weight=0.2, term_vals="filter"):
    """
    Run CellRank trajectory analysis for a specific genotype.
    
    Parameters
    ----------
    adata_orig : AnnData
        Full annotated AnnData object
    genotype : str
        "wt" or "het"
    conn_weight : float
        Weight on gene similarity (1 - conn_weight = weight on time coupling)
    term_vals : str
        Terminal state selection: "none", "filter", or "all"
        
    Returns
    -------
    dict
        Mean fate probabilities: {timepoint: {initial_EC: {terminal_EC: probability}}}
    """
    print(f"\n{'='*60}")
    print(f"Running CellRank for: {genotype.upper()}")
    print(f"conn_weight={conn_weight}, term_vals={term_vals}")
    print(f"{'='*60}")
    
    # Partition data by genotype
    # Group: 0=WT_Week0, 1=Het_Week0, 2=WT_Week12, 3=Het_Week12
    adata_wt0 = adata_orig[adata_orig.obs["group"] == 0].copy()
    adata_het0 = adata_orig[adata_orig.obs["group"] == 1].copy()
    adata_wt12 = adata_orig[adata_orig.obs["group"] == 2].copy()
    adata_het12 = adata_orig[adata_orig.obs["group"] == 3].copy()
    
    # Combine timepoints for selected genotype
    if genotype == "wt":
        adata = ad.concat([adata_wt0, adata_wt12], join="outer")
    elif genotype == "het":
        adata = ad.concat([adata_het0, adata_het12], join="outer")
    else:
        raise ValueError('genotype must be "wt" or "het"')
    
    print(f"Combined data: {adata.n_obs} cells")
    
    # Store raw counts
    adata.raw = adata
    
    # Create time key from sample identifier
    # Extract timepoint: 0 for Week0, 12 for Week12
    adata.obs["time_key"] = [
        0.0 if "Week0" in str(ident) else 12.0 
        for ident in adata.obs["orig.ident"]
    ]
    adata.obs["time_key"] = adata.obs["time_key"].astype("category")
    adata.obs["subcluster_annotation"] = adata.obs["subcluster_annotation"].astype("category")
    
    print(f"Time distribution: {adata.obs['time_key'].value_counts().to_dict()}")
    
    # -------------------------------------------------------------------------
    # Run moscot optimal transport
    # -------------------------------------------------------------------------
    print("\nRunning moscot optimal transport...")
    
    tp = TemporalProblem(adata)
    tp = tp.score_genes_for_marginals(
        gene_set_proliferation="mouse",
        gene_set_apoptosis="mouse"
    )
    tp = tp.prepare(time_key="time_key")
    tp = tp.solve(epsilon=1e-3, tau_a=0.95, scale_cost="mean")
    
    # -------------------------------------------------------------------------
    # Compute transition matrix
    # -------------------------------------------------------------------------
    print("Computing transition matrix...")
    
    tmk = RealTimeKernel.from_moscot(tp)
    tmk.compute_transition_matrix(
        self_transitions="all",
        conn_weight=conn_weight,
        threshold="auto"
    )
    
    # -------------------------------------------------------------------------
    # Initial fate probability estimation
    # -------------------------------------------------------------------------
    print("Computing initial fate probabilities...")
    
    g = cr.estimators.GPCCA(tmk, verbose=False)
    g.compute_schur(verbose=False)
    
    num_states = 5  # Number of EC subtypes
    g.compute_macrostates(
        n_states=num_states,
        n_cells=30,
        cluster_key="subcluster_annotation"
    )
    
    g.set_terminal_states(
        states=["0", "1", "2", "3", "4"],
        n_cells=30,
        cluster_key="subcluster_annotation",
        allow_overlap=True
    )
    
    g.compute_fate_probabilities(show_progress_bar=False)
    init_fate_probs = np.array(g.fate_probabilities)
    
    # -------------------------------------------------------------------------
    # Set terminal states based on term_vals parameter
    # -------------------------------------------------------------------------
    terminal_clusters = list(adata.obs["subcluster_annotation"])
    
    if term_vals == "all":
        # Set all Week12 cells as terminal
        for i in range(len(terminal_clusters)):
            if adata.obs["time_key"].iloc[i] != 12.0:
                terminal_clusters[i] = None
    else:
        # Use auto-detected terminal states
        for i in range(len(terminal_clusters)):
            if sum(init_fate_probs[i]) != 1:
                terminal_clusters[i] = None
            if term_vals == "filter":
                if adata.obs["time_key"].iloc[i] != 12.0:
                    terminal_clusters[i] = None
    
    terminal_clusters = pd.Series(terminal_clusters, dtype="category")
    terminal_clusters.index = adata.obs["time_key"].index
    
    # -------------------------------------------------------------------------
    # Final fate probability calculation
    # -------------------------------------------------------------------------
    print("Computing final fate probabilities...")
    
    p = cr.estimators.GPCCA(tmk)
    p.compute_schur()
    p.set_terminal_states(
        states=terminal_clusters,
        cluster_key="subcluster_annotation",
        allow_overlap=True
    )
    p.compute_fate_probabilities(show_progress_bar=False)
    
    final_fate_probs = np.array(p.fate_probabilities)
    fprob_col = p.fate_probabilities.names
    
    # -------------------------------------------------------------------------
    # Calculate mean fate probabilities per initial subtype
    # -------------------------------------------------------------------------
    print("Calculating mean fate probabilities...")
    
    means_dict = {0: {}, 12: {}}  # {timepoint: {initial_EC: {terminal_EC: mean}}}
    
    for timepoint in [0, 12]:
        for i in range(5):
            ec_group = EC_LABELS[i]
            means_dict[timepoint][ec_group] = {}
            
            # Get cells for this timepoint
            fate_probs_tp = final_fate_probs[adata.obs["time_key"] == timepoint]
            subclusters_tp = adata.obs[adata.obs["time_key"] == timepoint]["subcluster_annotation"]
            
            # Get cells for this initial EC subtype
            mask = subclusters_tp == ec_group
            fate_probs_group = fate_probs_tp[mask]
            
            if len(fate_probs_group) == 0:
                for j in range(5):
                    means_dict[timepoint][ec_group][EC_LABELS[j]] = 0.0
                continue
            
            # Calculate mean probability for each terminal state
            for group_ind, terminal_ec in enumerate(fprob_col):
                try:
                    mean_prob = np.mean(fate_probs_group[:, group_ind])
                    means_dict[timepoint][ec_group][terminal_ec] = mean_prob
                except (KeyError, IndexError):
                    means_dict[timepoint][ec_group][terminal_ec] = 0.0
    
    print("Done!")
    return means_dict

## Run Analysis for WT and Het

In [ ]:
# Run CellRank for both genotypes
# Using recommended parameters: conn_weight=0.2, term_vals="filter"

wt_results = run_cellrank_analysis(
    adata_orig,
    genotype="wt",
    conn_weight=0.2,
    term_vals="filter"
)

het_results = run_cellrank_analysis(
    adata_orig,
    genotype="het",
    conn_weight=0.2,
    term_vals="filter"
)

## Calculate Het vs WT Delta

In [ ]:
def calculate_delta(wt_results, het_results, timepoint):
    """
    Calculate difference in fate probabilities: Het - WT
    
    Returns a DataFrame with initial EC as rows and terminal EC as columns.
    """
    ec_types = ["EC1.1", "EC1.2", "EC1.3", "EC2", "EC3"]
    delta_matrix = []
    
    for init_ec in ec_types:
        row = []
        for term_ec in ec_types:
            wt_prob = wt_results.get(timepoint, {}).get(init_ec, {}).get(term_ec, 0)
            het_prob = het_results.get(timepoint, {}).get(init_ec, {}).get(term_ec, 0)
            delta = het_prob - wt_prob
            row.append(delta)
        delta_matrix.append(row)
    
    return pd.DataFrame(delta_matrix, index=ec_types, columns=ec_types)

# Calculate deltas for both timepoints
delta_week0 = calculate_delta(wt_results, het_results, 0)
delta_week12 = calculate_delta(wt_results, het_results, 12)

print("Delta (Het - WT) at Week 0 → Week 12:")
print(delta_week0.round(3))
print("\nDelta (Het - WT) at Week 12 → Week 24:")
print(delta_week12.round(3))

## Visualize Results

In [ ]:
import seaborn as sns

def plot_fate_delta(delta_df, title, vmin=-0.35, vmax=0.35):
    """
    Plot heatmap of fate probability differences.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    
    sns.heatmap(
        delta_df,
        annot=True,
        fmt=".3f",
        cmap="RdBu_r",
        center=0,
        vmin=vmin,
        vmax=vmax,
        ax=ax,
        linewidths=0.5,
        linecolor="white"
    )
    
    ax.set_xlabel("Terminal EC State", fontsize=12, fontweight="bold")
    ax.set_ylabel("Initial EC State", fontsize=12, fontweight="bold")
    ax.set_title(title, fontsize=14, fontweight="bold")
    
    plt.tight_layout()
    return fig

# Plot deltas
fig1 = plot_fate_delta(
    delta_week0,
    "Fate Probability Delta (Het - WT)\nWeek 0 → Week 12"
)
plt.show()

fig2 = plot_fate_delta(
    delta_week12,
    "Fate Probability Delta (Het - WT)\nWeek 12 → Week 24"
)
plt.show()

## Save Results

In [ ]:
import os

# Save to Excel
output_path = os.path.join(output_dir, output_file)

with pd.ExcelWriter(output_path) as writer:
    # WT results
    pd.DataFrame(wt_results[0]).to_excel(writer, sheet_name="WT_Week0")
    pd.DataFrame(wt_results[12]).to_excel(writer, sheet_name="WT_Week12")
    
    # Het results
    pd.DataFrame(het_results[0]).to_excel(writer, sheet_name="Het_Week0")
    pd.DataFrame(het_results[12]).to_excel(writer, sheet_name="Het_Week12")
    
    # Delta matrices
    delta_week0.to_excel(writer, sheet_name="Delta_W0_to_W12")
    delta_week12.to_excel(writer, sheet_name="Delta_W12_to_W24")

print(f"Results saved to: {output_path}")

## Output

The Excel file contains:
- **WT_Week0**: WT fate probabilities from Week 0 initial states
- **WT_Week12**: WT fate probabilities from Week 12 initial states  
- **Het_Week0**: Het fate probabilities from Week 0 initial states
- **Het_Week12**: Het fate probabilities from Week 12 initial states
- **Delta_W0_to_W12**: Het - WT difference for Week 0 → Week 12 transition
- **Delta_W12_to_W24**: Het - WT difference for Week 12 → Week 24 transition

The delta matrices are visualized in `02_EC_subclustering.Rmd` and can be used to create Sankey diagrams with `05_cellrank_sankey.Rmd`.